<div style="background-color:#0B3C5D; padding:25px; border-radius:10px;">

![](https://raw.githubusercontent.com/wateraccounting/WaPORMOOC/main/images/banner_notebooks_WaPOR4Global.png)


<div style="text-align:center; margin-top:15px;">
<h3 style="color:#D9E6F2; margin-bottom:5px;">
<i>Module Three</i> – Topic 1a - <b>Characterising Iraq's Climate</b>
</h3>

<p style="color:white; margin-top:5px;">
<b>Estimated time:</b> 2 hours &nbsp; | &nbsp;
<b>Instructor:</b> Dr. Ahmed Elnaggar
</p>
</div>

</div>


---

### 📚 Course Information

| Item | Description |
|------|-------------|
| **Course** | MOOC — WaPOR for Global Challenges |
| **Module** | 3: WaPOR analyses |
| **Topic** | 1 of 3: Long-Term Time Series Analysis |
| **Focus** | Spatial and temporal variability of Rainfall & Reference ET |
| **Region** | Iraq — 18 Governorates |
| **Data** | Monthly observations (1980–present) |

---

### 🎯 Learning Objectives

By the end of this notebook, you will be able to:

1. **Characterize** the climatological patterns of rainfall and reference ET across Iraq
2. **Identify** drought and wet anomalies at the governorate level
3. **Compute** aridity indices to assess regional water stress

---

### 📖 Table of Contents

1. Module 3 WaPOR analyses overview
2. Iraq's Water Landscape
3. Setup & Data Loading
4. Descriptive Statistics
5. Temporal Patterns — The Big Picture
6. The Clock of Seasons — Monthly Climatology
7. Are Things Changing? — Trend Analysis
8. When Extremes Strike — Anomaly Analysis
9. The Aridity Lens — Water Supply vs. Demand
10. Summary

---

## 1. Module 3 WaPOR analyses overview

This notebook is part of the **first of three topics** in the module 'WaPOR analyses' of the MOOC on *"WaPOR for Global Challenges"*. Together, they tell a story from the national to the field level using Iraq as a case study:

> 🌍 **Topic 1 — The Wide Lens (This Notebook):**
> We begin at the **country level**, analysing long-term rainfall and reference evapotranspiration across all **18 governorates** of Iraq. This establishes the climatic baseline: *Where is it wet? Where is it dry? Is the climate changing?*

> 🏙️ **Topic 2 — Zooming Into Erbil (Notebook can be found here):**
> We narrow our focus to a single governorate — **Erbil** — using high-resolution **FAO WaPOR data**. Here we analyse drought dynamics, water productivity, and water-use patterns and their interactions at much finer spatial and temporal detail.

> 🌾 **Topic 3 — Down to the Field (Notebook can be found here):**
> Finally, we zoom to the **field level**, selecting ~13 centre-pivot irrigated fields near Erbil. We apply **ARIMA and SARIMAX forecasting models** to predict water consumption, comparing performance across individual fields.

Each step builds on the previous one — understanding the country’s climate helps interpret what we see in Erbil, and understanding Erbil helps frame why field-level forecasting matters for operational water management.

---

## 2. Iraq’s Water Landscape

Iraq, the land of **Mesopotamia** ("between the rivers"), has been shaped by the Tigris and Euphrates for millennia. But today, this cradle of civilisation faces an unprecedented water crisis affecting its agricultural sector:

| Challenge | Description |
|-----------|-------------|
| 🌡️ **Rising temperatures** | Average temperatures have increased by ~1.5°C since the 1970s |
| 🌧️ **Declining rainfall** | 10–20% reduction in annual precipitation in some regions |
| 🏗️ **Upstream dams** | Turkey and Iran’s dam construction has reduced river flows |
| 🌾 **Agricultural demand** | ~80% of water use goes to agriculture |
| 📈 **Population growth** | Doubling since 1990, increasing water demand |


---


> 💡 **Key Questions:** *What is the spatio-temporal variability of the climate in Iraq? How is the climate changing over time?*



---

## 3. Setting up the environment & Load data for analyses

### Required Packages

We use standard Python data-science libraries.

In [2]:
# =============================================================================
# INSTALL & IMPORT PACKAGES
# =============================================================================

# Uncomment to install if needed:
!pip install numpy pandas matplotlib seaborn scipy pymannkendall --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from scipy import stats
import os
import warnings
from datetime import datetime
from pathlib import Path

# Mann-Kendall trend testing
try:
    import pymannkendall as mk
    MK_AVAILABLE = True
    print("\u2705 pymannkendall loaded")
except ImportError:
    MK_AVAILABLE = False
    print("\u26a0\ufe0f pymannkendall not installed \u2014 using scipy fallback for trends")

# --- Configuration ---
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = [14, 7]
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print(f"\n\u2705 Setup complete \u2014 Pandas {pd.__version__}, NumPy {np.__version__}")
print(f"\U0001f4c5 Analysis date: {datetime.now().strftime('%Y-%m-%d')}")

✅ pymannkendall loaded

✅ Setup complete — Pandas 2.2.2, NumPy 2.0.2
📅 Analysis date: 2026-04-10


### Data Files
Create folders for where to upload the input data and where to store the results.


In [ ]:
# =============================================================================
# CREATE INPUT AND OUTPUT FOLDERS
# =============================================================================

data_folder = 'data'
os.makedirs(data_folder, exist_ok=True)

output_folder = 'output_data'
os.makedirs(output_folder, exist_ok=True)

### Upload the data files
Place the following data* files in a new folder called `data/` (source of data: AgERA5 for the period 1980-2025):

| File | Description |
|------|-------------|
| `AgERA5-PCP-M_mm.csv` | Monthly rainfall (mm/month) for 18 governorates of Iraq|
| `AgERA5-RET-M_mm.csv` | Monthly reference ET (mm/month) for 18 governorates of Iraq |

**Expected CSV format:** First column = date (`DD/MM/YYYY`), remaining columns = governorate names.

*Instructions on how to download the required data is provided in this [Notebook](notebook link).

### Define data format
First the format of the data needs to be defined. In the following script, the
CSV file is read from a file path, different dates formats are recognised and the first column is idenfitied as date and it is sorted.

In [ ]:
# =============================================================================
# DEFINING DATA FORMAT
# =============================================================================

def load_governorate_data(filepath, name='dataset'):
    """
    Load monthly governorate-level data from CSV.
    Expects: first column = date, remaining columns = governorates.
    """
    # Added 'delimiter=';'' to handle semicolon-separated CSVs
    df = pd.read_csv(filepath, delimiter=';')
    date_col = df.columns[0]

    # Parse dates (try multiple formats)
    for fmt in ['%d/%m/%Y', '%m/%d/%Y', '%Y-%m-%d', '%Y/%m/%d']:
        try:
            df[date_col] = pd.to_datetime(df[date_col], format=fmt)
            break
        except (ValueError, TypeError):
            continue
    else:
        df[date_col] = pd.to_datetime(df[date_col], dayfirst=True)

    df.set_index(date_col, inplace=True)
    df.index.name = 'date'
    df = df.sort_index()

    # Ensure numeric
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    print(f"\U0001f4c4 {name}: {df.shape[0]} months × {df.shape[1]} governorates")
    print(f"   Period: {df.index.min().strftime('%b %Y')} — {df.index.max().strftime('%b %Y')}")
    return df


### Read data files
The `load_governorate_data` function is run using the following script for each data file. For this exercise you read monthly rainfall and reference ET data from AgERA5 for the period 1980-2025.

Check if for each data set you have data for the entire period.

In [ ]:
# =============================================================================
# READ DATA FILES
# =============================================================================

# --- Load Rainfall ---
rainfall_file = os.path.join(data_folder, '/content/data/AgERA5-PCP-M_mm.csv')
if os.path.exists(rainfall_file):
    df_rain = load_governorate_data(rainfall_file, name='Rainfall')
else:
    raise FileNotFoundError(f"Rainfall file not found: {rainfall_file}")

# --- Load Reference ET ---
ret_file = os.path.join(data_folder, '/content/data/AgERA5-RET-M_mm.csv')
if os.path.exists(ret_file):
    df_ret = load_governorate_data(ret_file, name='Reference ET')
else:
    raise FileNotFoundError(f"Reference ET file not found: {ret_file}")

# --- Governorate list ---
governorates = df_rain.columns.tolist()
n_gov = len(governorates)
print(f"\n\U0001f4cd {n_gov} governorates detected: {governorates}")

### Inspect the data

The previous script checks the data that is contained in the files. Inspect if the data contains information for all the 18 governorates and the period 1980-2025.

If all is in order, now lets have a look at the data!

The precipitation data file is loaded into the script as `df_rain/`. To view the content of the file, run the next cell.

***Exercise***: Can you also inspect the Reference ET data file?

HINT: Inspect the code in the previous cell and identify the name of the dataframe which contains the reference ET data.

In [ ]:
# =============================================================================
# DATA INSPECTION
# =============================================================================

print("RAINFALL DATA \u2014 First 12 Months")
display(df_rain.head(12))


### Quick check for completeness of the rainfall data

The following cell will check if the data has any missing values.

***Exercise:*** Can you check if the Reference ET data is also complete?

In [ ]:
# =============================================================================
# CHECK DATA FOR COMPLETENESS
# =============================================================================

print("\n\U0001f4ca Data Completeness:")
rain_missing = df_rain.isnull().sum()

if rain_missing.sum() == 0:
    print("   Rainfall:     \u2705 No missing values")
else:
    print(f"   Rainfall:     \u26a0\ufe0f {rain_missing.sum()} missing values")
    print(rain_missing[rain_missing > 0])

---

## 4. Temporal Patterns — The Big Picture

> *"Let’s unroll the tape and see four decades of climate history."*

We start with plotting the raw time series. Run the cell below to visualise the monthly rainfall for all governorates.

***Exercise:*** update the code to visualise the Reference ET data as well.

In [ ]:
# =============================================================================
# RAINFALL TIME SERIES - ALL GOVERNORATES IN ONE GRAPH
# =============================================================================

df_rain.plot(legend=False);


---

### 5. Descriptive Statistics

> *"Before we tell the story, let’s meet the characters."*

Each governorate has its own climate personality — some consistently wet, others bone-dry, some wildly variable from year to year. Descriptive statistics help us characterise these personalities before we look at trends or anomalies. In the following cell the most common statistics are calculated. `('YE').sum/` corresponds to summarising the data annually.

NOTE: If the data is provided in daily timesteps, this script can also be used to create monthly timeseries using `('ME').sum/` or from sub-daily to daily timeseries using `('DE').sum/`.




In [ ]:
# =============================================================================
# DESCRIPTIVE STATISTICS (ANNUAL TOTALS)
# =============================================================================

# Aggregate to annual totals
rain_annual = df_rain.resample('YE').sum()

# --- Rainfall statistics ---
rain_stats = pd.DataFrame({
    'Mean (mm/yr)': rain_annual.mean(),
    'Std (mm/yr)':  rain_annual.std(),
    'CV (%)':       (rain_annual.std() / rain_annual.mean()) * 100,
    'Min (mm/yr)':  rain_annual.min(),
    'Max (mm/yr)':  rain_annual.max(),
    'Median (mm/yr)': rain_annual.median(),
    'Skewness':     rain_annual.skew()
}).round(1)

print("="*70)
print("ANNUAL RAINFALL STATISTICS BY GOVERNORATE")
print("="*70)
display(rain_stats.sort_values('Mean (mm/yr)', ascending=False))

### Sorting the data frame
You can interrogate the `rain_stats` using different functions. The data can be sorted in ascending or descending order using the `sort_values` function.

***Excercise*** Which governorate has the lowest mean annual rainfall? What is the amount and standard deviation?

In [ ]:
# =============================================================================
# SORTING THE DATA FRAME
# =============================================================================

sorted_govs_rain = rain_stats.sort_values('Max (mm/yr)', ascending=False)
print(sorted_govs_rain)


### Returning the mean or maximum values
The following script returns 1) the maximum value and 2) the name of the governorate which recorded the maximum value.

In [ ]:
# =============================================================================
# IDENTIFYING THE GOVERNORATE WITH THE HIGHEST MEAN ANNUAL RAINFALL
# =============================================================================

print(rain_stats['Mean (mm/yr)'].max())
print(rain_stats['Mean (mm/yr)'].idxmax())
print('The governorate with the highest rainfall is',rain_stats['Mean (mm/yr)'].idxmax(),'with mean annual rainfall of', rain_stats['Mean (mm/yr)'].max(), 'mm/year')


879.5
Duhok
The governorate with the highest rainfall is Duhok with mean annual rainfall of 879.5 mm/year


***Exercise:*** Repeat the previous steps and calculate the statistics for the reference ET data.


In [ ]:
# =============================================================================
# DESCRIPTIVE STATISTICS (ANNUAL TOTALS)
# =============================================================================

# Aggregate to annual totals

# --- Reference ET statistics ---



## Quiz question:

Which governorate has the highest annual RET and what is the value (in mm/year - no digits)?
Which governorate has the lowest annual RET and what is the value (in mm/year- no digits)

Write down the answers, you need it for the quiz!

---

## 6. The Clock of Seasons — Monthly Climatology

> *"Every year, the same rhythm: rain falls in winter, the sun blazes in summer, and the land oscillates between brief abundance and long thirst."*

Monthly climatology — the average value for each month across all years — reveals the **seasonal heartbeat** of Iraq’s climate. This is the background pattern against which all variability and change are measured.

### Calculate the mean monthly values
First lets calculate the mean monthly values and assign the names of the month to the first column (index).

In [ ]:
# =============================================================================
# MONTHLY CLIMATOLOGY
# =============================================================================

month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

# Compute monthly climatology
rain_clim = df_rain.groupby(df_rain.index.month).mean()
rain_clim.index = month_names

print(rain_clim)

### Visualise the monthly values
Lets visualise this data using the line graph style. The graphs show the values according to the calendar (Jan-Dec). To update the graphs following the hydrological year (Sep-Aug), remove the comments under *"reorder months to start from September"* and rerun the cell.

In [ ]:
# =============================================================================
# VISUALISING MONTHLY RAINFALL
# =============================================================================

# Reorder months to start from September
#month_order_sept_aug = ['Sep', 'Oct', 'Nov', 'Dec', 'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug']
#rain_clim = rain_clim.reindex(month_order_sept_aug)
#month_names = month_order_sept_aug

fig, ax = plt.subplots(1, 1, figsize=(16, 6))

# Plot each governorate's monthly climatology as a line
rain_clim.plot(ax=ax, cmap='viridis', linewidth=2)

ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Average Rainfall (mm/month)', fontsize=12)
ax.set_title('Monthly Rainfall', fontsize=14, fontweight='bold')
ax.set_xticks(range(len(month_names)))
ax.set_xticklabels(month_names, rotation=45, ha='right')
ax.legend(title='Governorate', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout(rect=[0, 0.03, 0.85, 0.98]) # Adjust layout to make space for the legend
plt.show()

---

## 7. Are Things Changing? — Trend Analysis

> *"Is Iraq getting drier? Is evaporative demand increasing? These are not abstract questions — they determine whether future generations can farm the same land."*

We first plot the annual data for selected governorates located in different areas of Iraq (Duhok, Erbil and Baghdad).

In [ ]:
# =============================================================================
# ANNUAL RAINFALL
# =============================================================================
rain_annual = df_rain.resample('YE').sum()

selection = ['Duhok', 'Erbil', 'Baghdad']

fig, axes = plt.subplots(len(selection), 1,
                         figsize=(14, 3 * len(selection)), sharex=True)
fig.suptitle('Annual Rainfall \u2014 Selected Governorates',
             fontsize=15, fontweight='bold', y=1.01)

if len(selection) == 1:
    axes = [axes]

for idx, gov in enumerate(selection):
    ax = axes[idx]
    years = rain_annual.index.year
    values = rain_annual[gov].values

    ax.bar(years, values, color='steelblue', alpha=0.6, width=0.8)

    ax.set_ylabel('mm/yr')
    ax.set_title(gov, fontsize=11, fontweight='bold', loc='left')
    ax.legend(fontsize=9, loc='upper right')
    ax.grid(True, alpha=0.2, axis='y')

axes[-1].set_xlabel('Year')
plt.tight_layout()
plt.show()

### Trend analyses
We apply the **Mann-Kendall test** — a robust non-parametric method for detecting monotonic trends in time series — to the annual totals of each governorate. The associated **Sen’s slope** tells us the rate of change (mm/year).

| Test | Purpose | Null Hypothesis |
|------|---------|-----------------|
| **Mann-Kendall** | Detect monotonic trend | No trend exists |
| **Sen’s Slope** | Estimate trend magnitude | — |

A **p-value < 0.05** indicates a statistically significant trend.

The following code cell creates the function to calculate the Mann-Kendall test.

In [ ]:
# =============================================================================
# DEFINING FUNCTIONS FOR MANN-KENDALL TEST
# =============================================================================

def sens_slope(series):
    """Calculate Sen's slope estimate."""
    n = len(series)
    slopes = []
    values = series.values
    for i in range(n):
        for j in range(i + 1, n):
            slopes.append((values[j] - values[i]) / (j - i))
    return np.median(slopes)

def mann_kendall_test(series):
    """Perform Mann-Kendall trend test."""
    series_clean = series.dropna()
    if len(series_clean) < 10:
        return ('insufficient data', np.nan, np.nan)

    if MK_AVAILABLE:
        result = mk.original_test(series_clean)
        return (result.trend, result.p, result.slope)
    else:
        # Scipy fallback
        x = np.arange(len(series_clean))
        tau, p = stats.kendalltau(x, series_clean.values)
        slope = sens_slope(series_clean)
        if p < 0.05:
            trend = 'increasing' if tau > 0 else 'decreasing'
        else:
            trend = 'no trend'
        return (trend, p, slope)


### Slope and significance
In the following code cell you evaluate the slope and significance for the annual rainfall.

In [ ]:
# =============================================================================
# RUNNING MANN-KENDALL FUNCTION FOR RAINFALL
# =============================================================================

print("="*70)
print("RAINFALL TREND ANALYSIS (Annual Totals)")
print("="*70)

rain_trend_results = []
for gov in governorates:
    trend, p, slope = mann_kendall_test(rain_annual[gov])
    rain_trend_results.append({
        'Governorate': gov, 'Trend': trend,
        'p-value': round(p, 4), 'Slope (mm/yr)': round(slope, 2)
    })
rain_trend_df = pd.DataFrame(rain_trend_results).set_index('Governorate')
display(rain_trend_df.sort_values('Slope (mm/yr)'))

rain_sig = rain_trend_df[rain_trend_df['p-value'] < 0.05]
print(f"\n\U0001f4c8 Significant rainfall trends: {len(rain_sig)} / {n_gov}")

***Exercise:*** Repeat the steps for the reference ET data.

Quiz questions:

How many stations have a significant trend in RET?
Which governorate has the largest rate of change (slope) and how much is this in mm/year (2 digits)?

In [ ]:
# =============================================================================
# RUNNING MANN-KENDALL FUNCTION FOR RET
# =============================================================================


### Visualising trend analyses results
Lets visualise these results in the graphs. The script below is the same as before with additional lines to visualise the trendline and the statistical values.

In [ ]:
# =============================================================================
# TREND TIME SERIES - ANNUAL RAINFALL WITH FITTED LINE
# =============================================================================

fig, axes = plt.subplots(len(selection), 1,
                         figsize=(14, 3 * len(selection)), sharex=True)
fig.suptitle('Annual Rainfall with Trend Line \u2014 Selected Governorates',
             fontsize=15, fontweight='bold', y=1.01)

if len(selection) == 1:
    axes = [axes]

for idx, gov in enumerate(selection):
    ax = axes[idx]
    years = rain_annual.index.year
    values = rain_annual[gov].values

    ax.bar(years, values, color='steelblue', alpha=0.6, width=0.8)

    # Fit trend line
    slope_val = rain_trend_df.loc[gov, 'Slope (mm/yr)']
    p_val     = rain_trend_df.loc[gov, 'p-value']
    x_num = np.arange(len(years))
    intercept = np.median(values) - slope_val * np.median(x_num)
    trend_line = intercept + slope_val * x_num

    line_color = '#d62728' if slope_val < 0 else '#2ca02c'
    ax.plot(years, trend_line, color=line_color, linewidth=2.5, linestyle='--',
            label=f"Trend: {slope_val:+.1f} mm/yr (p={p_val:.3f})")

    ax.set_ylabel('mm/yr')
    ax.set_title(gov, fontsize=11, fontweight='bold', loc='left')
    ax.legend(fontsize=9, loc='upper right')
    ax.grid(True, alpha=0.2, axis='y')

axes[-1].set_xlabel('Year')
plt.tight_layout()
plt.show()

### Anomalies

> *"It’s not the average that breaks a farmer — it’s the year that deviates from it."*

**Standardised anomalies** measure how unusual a year is compared to the long-term average:

$$Z_i = \frac{X_i - \bar{X}}{\sigma_X}$$

Where $X_i$ is the annual rainfall in year $i$, $\bar{X}$ is the long-term mean, and $\sigma_X$ is the standard deviation.

In [ ]:
# =============================================================================
# CALCULATING STANDARDISED ANOMALY
# =============================================================================

# Compute standardised anomalies for annual rainfall
rain_anomaly = (rain_annual - rain_annual.mean()) / rain_annual.std()
# print(rain_anomaly)


### Drought classification
Let's classify the results according to the five categories and give each category a different color bar.


| Z value | Classification |
|---------|---------------|
| Z ≤ −2.0 | **Extreme drought** |
| −2.0 < Z ≤ −1.0 | **Moderate drought** |
| −1.0 < Z < 1.0 | **Near normal** |
| 1.0 ≤ Z < 2.0 | **Moderately wet** |
| Z ≥ 2.0 | **Extremely wet** |

This reveals *when* and *where* droughts and wet periods hit Iraq hardest.

In [ ]:
# =============================================================================
# ANNUAL ANOMALY BAR CHARTS - SELECTED GOVERNORATES
# =============================================================================

selection = ['Erbil']
fig, axes = plt.subplots(len(selection), 1,
                         figsize=(16, 3 * len(selection)), sharex=True)
fig.suptitle('Annual Rainfall Anomaly (Standardised)',
             fontsize=16, fontweight='bold', y=1.01)

if len(selection) == 1:
    axes = [axes]

for idx, gov in enumerate(selection):
    ax = axes[idx]
    anomaly = rain_anomaly[gov]
    years = anomaly.index.year

    # Colour by sign
    colors = ['#d62728' if v < -1 else '#2166ac' for v in anomaly.values]

    ax.bar(years, anomaly.values, color=colors, width=0.8,
           edgecolor='none', alpha=0.8)
    ax.set_ylabel('Z-score')
    ax.set_title(gov, fontsize=11, fontweight='bold', loc='left')
    ax.set_ylim(-3.5, 3.5)
    ax.grid(True, alpha=0.2, axis='y')

    # Count drought years
    n_drought = (anomaly < -1).sum()
    ax.text(0.98, 0.95, f'{n_drought} drought years (Z < -1)',
            transform=ax.transAxes, fontsize=9, ha='right', va='top',
            color='#d62728', fontweight='bold')

axes[-1].set_xlabel('Year')
plt.tight_layout()
plt.show()


---

## 8. The Aridity Lens — Water Supply vs. Demand

> *"The true measure of water stress isn’t just how little rain falls — it’s how much water the atmosphere demands in return."*

The **UNEP Aridity Index** combines both rainfall and reference ET into a single metric:

$$AI = \frac{P}{ET_0}$$


Let's calculated the aridity index for each governorate using the mean annual Precipitation value divided by the mean annual reference ET value. Now remove the # before `.sort_values` and see what happens.

In [ ]:
# =============================================================================
# ARIDITY INDEX - P / ET0
# =============================================================================

# Compute annual aridity index
common_govs = [g for g in governorates if g in df_refet.columns]
refet_annual = df_refet.resample('YE').sum()

ai_annual = rain_annual[common_govs] / refet_annual[common_govs]
ai_mean = ai_annual.mean()#.sort_values(ascending=False)
print(ai_mean)


### Visualisation of Aridity index
Create a simple bar chart to visualise the results.

In [ ]:
# =============================================================================
# VISUALISING ARIDITY INDEX PER GOVERNORATE
# =============================================================================

fig, axes = plt.subplots(1, 1, figsize=(16, max(6, n_gov * 0.4)))
fig.suptitle('Aridity Index (P / ET\u2080) \u2014 Long-Term Average',
             fontsize=16, fontweight='bold')

ax1 = axes
ax1.barh(ai_mean.index, ai_mean.values, color='orange', edgecolor='white')
ax1.set_xlabel('Aridity Index (P/ET\u2080)')
ax1.set_title('By Governorate', fontweight='bold')

plt.tight_layout()
plt.show()

### Aridity classification
Let's classify the results according to the five categories and give each category a different color bar.

| Aridity Index | Classification | Character |
|:---:|---|---|
| AI < 0.03 | **Hyper-arid** | Desert; virtually no vegetation |
| 0.03 – 0.20 | **Arid** | Very limited rain; irrigation essential |
| 0.20 – 0.50 | **Semi-arid** | Rainfed possible in good years |
| 0.50 – 0.65 | **Dry sub-humid** | Rainfed agriculture viable |
| > 0.65 | **Humid** | Adequate water supply |


In [ ]:
# Classification function
def classify_aridity(ai):
    if ai < 0.03:  return 'Hyper-arid'
    elif ai < 0.20: return 'Arid'
    elif ai < 0.50: return 'Semi-arid'
    elif ai < 0.65: return 'Dry sub-humid'
    else:            return 'Humid'

# --- Visualisation ---
fig, axes = plt.subplots(1, 1, figsize=(16, max(6, n_gov * 0.4)))
fig.suptitle('Aridity Index (P / ET\u2080) \u2014 Long-Term Average',
             fontsize=16, fontweight='bold')

# Bar chart
ax1 = axes
ai_colors_map = {
    'Hyper-arid': '#8B0000', 'Arid': '#d62728',
    'Semi-arid': '#ff7f0e', 'Dry sub-humid': '#2ca02c', 'Humid': '#1f77b4'
}
bar_colors = [ai_colors_map[classify_aridity(v)] for v in ai_mean.values]
ax1.barh(ai_mean.index, ai_mean.values, color=bar_colors, edgecolor='white')
ax1.set_xlabel('Aridity Index (P/ET\u2080)')
ax1.set_title('By Governorate', fontweight='bold')

# Classification thresholds
for threshold, label in [(0.03, 'Hyper-arid'), (0.20, 'Arid'),
                         (0.50, 'Semi-arid'), (0.65, 'Sub-humid')]:
    ax1.axvline(x=threshold, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

### Aridity summary

In [ ]:
# Classification summary
print("\U0001f4ca Aridity Classification Summary:")
print("-"*50)
ai_classes = pd.Series({g: classify_aridity(v) for g, v in ai_mean.items()})
for cls in ['Humid', 'Dry sub-humid', 'Semi-arid', 'Arid', 'Hyper-arid']:
    govs_in_class = ai_classes[ai_classes == cls].index.tolist()
    if govs_in_class:
        print(f"   {cls:15s}: {', '.join(govs_in_class)}")

---

## 9. Summary Key Findings

- exporting dfs to be used in visualisation section
- append data to csvs to be used invisualisation section

### Saving results as csv file
You can also save the data frames as csv files by using the following code. You need this to run the visualisation scripts!!

In [ ]:
# =============================================================================
# Saving results
# =============================================================================

#saving results rainfall analyses
filepath = Path("/content/output_data/rain_annual.csv")
rain_annual.to_csv(filepath)
filepath = Path("/content/output_data/rain_stats.csv")
rain_stats.to_csv(filepath)
filepath = Path("/content/output_data/rain_clim.csv")
rain_clim.to_csv(filepath)
filepath = Path("/content/output_data/rain_trend.csv")
rain_trend_df.to_csv(filepath)

#saving results reference et analyses
filepath = Path("/content/output_data/ret_annual.csv")
ret_annual.to_csv(filepath)
filepath = Path("/content/output_data/ret_stats.csv")
ret_stats.to_csv(filepath)
filepath = Path("/content/output_data/ret_clim.csv")
ret_clim.to_csv(filepath)
filepath = Path("/content/output_data/ret_trend.csv")
ret_trend_df.to_csv(filepath)

#saving results aridity index
filepath = Path("/content/output_data/ai_mean.csv")
ai_mean.to_csv(filepath)